### <h4 style="color:blue;">01 dependencies</h5>

In [ ]:
# standard library
import os
import random
from collections import Counter

# data handling
import pandas as pd

# image processing
from PIL import Image
import cv2

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# deep learning
import torch
from facenet_pytorch import MTCNN

# utilities
from tqdm import tqdm

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [ ]:
# base path
TRAIN_PATH = "../data/train"
TEST_PATH  = "../data/test"
input_dir = '../data/train'
output_dir = '../data/cropped'

### <h4 style="color:blue;">02 cropping image</h5>

In [ ]:
# Inisialisasi MTCNN
mtcnn = MTCNN(
    image_size=224,
    keep_all=False,       
    select_largest=True,  
    margin=60,            
    post_process=False,   
    device=device
)

> train

In [26]:
classes = ['realperson', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_mannequin', 'fake_unknown']

def center_crop(img, size=224):
    w, h = img.size
    min_dim = min(w, h)

    left = (w - min_dim) // 2
    top = (h - min_dim) // 2
    right = left + min_dim
    bottom = top + min_dim

    img = img.crop((left, top, right, bottom))
    return img.resize((size, size))

for cls in classes:
    os.makedirs(os.path.join(output_dir, cls), exist_ok=True)

total_images = 0
successful_crops = 0
fallback_used = 0

for cls in classes:
    class_input_path = os.path.join(input_dir, cls)
    class_output_path = os.path.join(output_dir, cls)
    
    if not os.path.exists(class_input_path):
        print(f"Folder {cls} tidak ditemukan, skip.")
        continue
        
    images = os.listdir(class_input_path)
    print(f"\nMemproses: {cls} ({len(images)} gambar)")
    
    for img_name in tqdm(images):
        total_images += 1
        img_path = os.path.join(class_input_path, img_name)
        out_path = os.path.join(class_output_path, img_name)
        
        try:
            img = Image.open(img_path).convert('RGB')

            # === MTCNN ===
            face = mtcnn(img)

            if face is not None:
                # save tensor ke image
                face_img = Image.fromarray(face.permute(1,2,0).byte().cpu().numpy())
                face_img.save(out_path)
                successful_crops += 1

            else:
                # === FALLBACK CENTER CROP ===
                fallback_img = center_crop(img, size=224)
                fallback_img.save(out_path)
                fallback_used += 1

        except Exception as e:
            # fallback kalau error juga
            fallback_img = center_crop(img, size=224)
            fallback_img.save(out_path)
            fallback_used += 1

print("\n" + "="*30)
print("Ringkasan Cropping")
print(f"Total gambar: {total_images}")
print(f"MTCNN berhasil: {successful_crops}")
print(f"Fallback (center crop): {fallback_used}")
print("="*30)


Memproses: realperson (425 gambar)


100%|██████████| 425/425 [03:23<00:00,  2.09it/s]



Memproses: fake_mask (289 gambar)


100%|██████████| 289/289 [02:17<00:00,  2.10it/s]



Memproses: fake_printed (123 gambar)


...


Memproses: fake_mannequin (207 gambar)


100%|██████████| 207/207 [00:46<00:00,  4.44it/s]



Memproses: fake_unknown (361 gambar)


100%|██████████| 361/361 [00:38<00:00,  9.43it/s]


Ringkasan Cropping
Total gambar: 1649
MTCNN berhasil: 1525
Fallback (center crop): 124


> test

In [32]:
# base path
input_test_dir = '../data/test'
output_test_dir = '../data/cropped_test'

In [34]:
os.makedirs(output_test_dir, exist_ok=True)

total_images = 0
successful_crops = 0
fallback_used = 0

def center_crop(img, size=224):
    w, h = img.size
    min_dim = min(w, h)

    left = (w - min_dim) // 2
    top = (h - min_dim) // 2
    right = left + min_dim
    bottom = top + min_dim

    img = img.crop((left, top, right, bottom))
    return img.resize((size, size))

images = os.listdir(input_test_dir)
print(f"Total gambar test: {len(images)}")

for img_name in tqdm(images):
    img_path = os.path.join(input_test_dir, img_name)
    out_path = os.path.join(output_test_dir, img_name)

    # skip kalau bukan image
    if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    total_images += 1

    try:
        img = Image.open(img_path).convert('RGB')

        # === MTCNN ===
        face = mtcnn(img)

        if face is not None:
            # save tensor ke image
            face_img = Image.fromarray(face.permute(1,2,0).byte().cpu().numpy())
            face_img.save(out_path)
            successful_crops += 1

        else:
            # === FALLBACK CENTER CROP ===
            fallback_img = center_crop(img, size=224)
            fallback_img.save(out_path)
            fallback_used += 1

    except Exception as e:
        fallback_img = center_crop(img, size=224)
        fallback_img.save(out_path)
        fallback_used += 1

print("\n" + "="*30)
print("Ringkasan Cropping (TEST)")
print(f"Total gambar: {total_images}")
print(f"MTCNN berhasil: {successful_crops}")
print(f"Fallback (center crop): {fallback_used}")
print("="*30)

Total gambar test: 404


100%|██████████| 404/404 [03:00<00:00,  2.23it/s]


Ringkasan Cropping (TEST)
Total gambar: 404
MTCNN berhasil: 382
Fallback (center crop): 22
